# Gout LLMOps — C07

Chạy trọn **GOUT_MT_001 T1 → T2 → T3**. Nếu một lượt lỗi, các lượt sau phải `skipped`. Notebook cần GPU.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
print('Uploaded:', zip_name)


In [ ]:
import zipfile, os, pathlib, shutil
work=pathlib.Path('/content/gout_c07')
if work.exists(): shutil.rmtree(work)
work.mkdir()
with zipfile.ZipFile(zip_name) as z: z.extractall(work)
root=work/'gout_units_v01'
os.chdir(root)
print(os.getcwd())


In [ ]:
!python -m pip install -q -r requirements-unit02.txt


In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật GPU trong Colab')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!python -m unittest discover -s tests -p 'test_*.py' -v


In [ ]:
from pathlib import Path
from unit03_chat_engine import select_multi_case_turns
turns=select_multi_case_turns(Path('examples/unit01/questions.jsonl'),'GOUT_MT_001',count=3)
for t in turns: print(t['sample_id'], '=>', t['user'])
assert [t['turn_id'] for t in turns]==[1,2,3]


In [ ]:
from unit03_chat_engine import run_three_turns
out=Path('runs/c07_real')
if out.exists(): shutil.rmtree(out)
result=run_three_turns(Path('examples/unit01/questions.jsonl'),'GOUT_MT_001',Path('configs/unit03_hf_locked_from_b.json'),Path('prompts/system_vi_v1.txt'),out)
print(result['manifest'])


In [ ]:
import json
manifest=json.loads(Path('runs/c07_real/manifest.json').read_text(encoding='utf-8'))
rows=[json.loads(x) for x in Path('runs/c07_real/predictions.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
print('status:',manifest['status'])
print('statuses:',[r['status'] for r in rows])
print('c07_structure_verified:',manifest['c07_structure_verified'])
print('c07_successful_three_turn_run:',manifest['c07_successful_three_turn_run'])
for r in rows:
 print('\n',r['sample_id'],r['status'],'history_turns=',r.get('call',{}).get('history_turns'))
 if r['status']=='ok': print(r['answer'])
 else: print(r.get('error') or r.get('skip_reason'))
assert len(rows)==3
assert [r['turn_id'] for r in rows]==[1,2,3]
assert manifest['c07_structure_verified'] is True
assert [r['status'] for r in rows]==['ok','ok','ok'], 'Nếu có error, các lượt sau phải skipped; gửi ZIP để kiểm tra.'
assert manifest['c07_successful_three_turn_run'] is True


In [ ]:
t1,t2,t3=rows
assert t2['history_before'][-1]['content']==t1['answer']
assert t3['history_before'][-1]['content']==t2['answer']
assert t3['call']['history_turns']==2
print('C07 REAL OK: T3 nhìn thấy đủ hai lượt trước bằng output model thật.')


In [ ]:
archive=shutil.make_archive('/content/Gout_C07_real_result','zip','runs/c07_real')
print(archive)
files.download(archive)


Gửi lại `Gout_C07_real_result.zip` để mình nghiệm thu C07 trước khi chuyển sang khối D (RAG).
